# 07 - Forward Model v2: Advanced Physics Features Test

This notebook tests whether a small set of additional physics-inspired features can improve the current forward prediction model.

The goal is **not** to replace the current final model automatically. Instead, we compare:

1. **Base v1 features** used in the current final forward model.
2. **Advanced v2 features** added on top of the existing physics features.
3. **Advanced v2 + log target transform** for distance targets.
4. Optional linear baselines with Ridge / Lasso / ElasticNet.

We decide whether to keep the current v1 model or move to a v2 model based on:

- MAE
- RMSE
- R²
- normalized MAE / RMSE
- P90/P95 absolute error
- near-feasible-zone performance
- constraint feasibility metrics

The most important point: we compare the current full feature pipeline against the same pipeline plus new features. This avoids an unfair comparison between raw features and advanced features.

## 1. Imports and Project Paths

In [1]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FORWARD_DIR = PROJECT_ROOT / "data" / "raw" / "forward_prediction"
INVERSE_DIR = PROJECT_ROOT / "data" / "raw" / "inverse_design"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
MODELS_DIR = OUTPUTS_DIR / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

for path in [SUBMISSIONS_DIR, MODELS_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Forward data directory:", FORWARD_DIR)
print("Inverse design directory:", INVERSE_DIR)
print("Reports directory:", REPORTS_DIR)

Project root: /home/alouiyaz/projects/boom-challenge-ejecta-prediction
Forward data directory: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/data/raw/forward_prediction
Inverse design directory: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/data/raw/inverse_design
Reports directory: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports


## 2. Load Data and Constraints

The constraints are used only for diagnostics. The forward model is trained from `train.csv` and `train_labels.csv`.

In [2]:
input_cols = [
    "energy", "angle_rad", "coupling", "strength",
    "porosity", "gravity", "atmosphere", "shape_factor"
]

target_cols = [
    "P80", "fines_frac", "oversize_frac",
    "R95", "R50_fines", "R50_oversize"
]

fragmentation_targets = ["P80", "fines_frac", "oversize_frac"]
distance_targets = ["R95", "R50_fines", "R50_oversize"]

raw_train = pd.read_csv(FORWARD_DIR / "train.csv")[input_cols]
raw_test = pd.read_csv(FORWARD_DIR / "test.csv")[input_cols]
y = pd.read_csv(FORWARD_DIR / "train_labels.csv")[target_cols]

constraints_path = INVERSE_DIR / "constraints.json"
if not constraints_path.exists():
    constraints_path = PROJECT_ROOT / "constraints.json"

if constraints_path.exists():
    with open(constraints_path, "r") as f:
        constraints_data = json.load(f)
    output_constraints = constraints_data["constraints"]
    input_bounds = constraints_data["input_bounds"]
else:
    output_constraints = {"p80_min": 96.0, "p80_max": 101.0, "r95_max": 175.0}
    input_bounds = {}

p80_min = output_constraints["p80_min"]
p80_max = output_constraints["p80_max"]
r95_max = output_constraints["r95_max"]
p80_center = (p80_min + p80_max) / 2

print("Raw train:", raw_train.shape)
print("Raw test:", raw_test.shape)
print("Targets:", y.shape)
print("Constraints:")
display(pd.DataFrame([output_constraints]))

display(raw_train.head())
display(y.head())

Raw train: (2930, 8)
Raw test: (492, 8)
Targets: (2930, 6)
Constraints:


,p80_min,p80_max,r95_max
0,96.0,101.0,175.0


,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,3.826405,0.818303,0.861258,1.305809,0.337215,3.71,0.781263,0.784028
1,2.828754,1.193036,0.561245,3.494501,0.058029,1.62,0.136205,0.922737
2,3.068907,0.605872,0.948860,1.366386,0.315632,3.71,0.774704,0.954922
3,2.700574,1.073708,0.713705,3.599419,0.033062,1.62,0.144204,0.932911
4,3.484022,0.863568,1.237205,1.996742,0.278207,9.81,0.414620,1.260855


,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,76.972350,0.184728,0.016671,198.938699,175.527939,76.235779
1,269.057465,0.000622,0.916734,239.268477,447.157838,141.894047
2,104.070923,0.070343,0.094438,192.986417,189.286407,84.235774
3,257.618403,0.001026,0.880122,289.289693,500.000028,169.866473
4,111.717167,0.058576,0.136166,94.229304,97.614864,42.928393


## 3. Base Physics Feature Engineering

This function must match the current forward-model notebooks. It creates 48 columns from the 8 raw inputs.

In [3]:
def add_physics_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    eps = 1e-9

    # 1. Energy transfer features
    X["effective_energy"] = X["energy"] * X["coupling"]
    X["log_energy"] = np.log1p(X["energy"])
    X["log_effective_energy"] = np.log1p(X["effective_energy"])

    # 2. Angle decomposition
    X["sin_angle"] = np.sin(X["angle_rad"])
    X["cos_angle"] = np.cos(X["angle_rad"])
    X["tan_angle"] = np.tan(X["angle_rad"])
    X["horizontal_energy"] = X["effective_energy"] * X["cos_angle"]
    X["vertical_energy"] = X["effective_energy"] * X["sin_angle"]
    X["vertical_horizontal_ratio"] = X["vertical_energy"] / (X["horizontal_energy"] + eps)

    # 3. Material / fragmentation proxies
    X["energy_per_strength"] = X["energy"] / (X["strength"] + eps)
    X["effective_energy_per_strength"] = X["effective_energy"] / (X["strength"] + eps)
    X["material_resistance_index"] = X["strength"] * (1 - X["porosity"])
    X["fragmentation_index"] = X["effective_energy"] * X["porosity"] / (X["strength"] + eps)
    X["coupling_porosity"] = X["coupling"] * X["porosity"]
    X["coupling_atmosphere"] = X["coupling"] * X["atmosphere"]
    X["porosity_strength"] = X["porosity"] * X["strength"]

    # 4. Gravity and range proxies
    X["energy_per_gravity"] = X["energy"] / (X["gravity"] + eps)
    X["effective_energy_per_gravity"] = X["effective_energy"] / (X["gravity"] + eps)
    X["horizontal_energy_per_gravity"] = X["horizontal_energy"] / (X["gravity"] + eps)
    X["vertical_energy_per_gravity"] = X["vertical_energy"] / (X["gravity"] + eps)

    # 5. Atmosphere and drag proxies
    X["drag_proxy"] = X["atmosphere"] * X["shape_factor"]
    X["drag_per_gravity"] = X["drag_proxy"] / (X["gravity"] + eps)
    X["atmosphere_shape_energy"] = X["atmosphere"] * X["shape_factor"] * X["effective_energy"]
    X["atmosphere_per_gravity"] = X["atmosphere"] / (X["gravity"] + eps)

    # 6. Pi-like scaling proxies inspired by dimensional analysis
    X["pi_gravity_proxy"] = (X["gravity"] * X["coupling"]) / (X["energy"] + eps)
    X["pi_strength_proxy"] = X["strength"] / (X["gravity"] * X["coupling"] + eps)
    X["pi_atmosphere_proxy"] = X["atmosphere"] / (X["gravity"] * X["coupling"] + eps)

    # 7. Regime indicators observed during EDA
    X["porosity_regime"] = (X["porosity"] > 0.15).astype(int)
    X["strength_regime"] = (X["strength"] > 2.6).astype(int)
    X["angle_regime"] = (X["angle_rad"] > 0.95).astype(int)
    X["atm_regime"] = (X["atmosphere"] > 0.30).astype(int)
    X["regime_combo"] = (
        X["porosity_regime"] * 8
        + X["strength_regime"] * 4
        + X["angle_regime"] * 2
        + X["atm_regime"]
    )

    # 8. Additional cross-regime proxies
    X["scaled_energy"] = X["effective_energy"] / (X["strength"] * np.sqrt(X["gravity"]) + eps)
    X["fragility"] = X["porosity"] / (X["strength"] + eps)
    X["range_proxy"] = (X["effective_energy"] * (X["cos_angle"] ** 2)) / (X["gravity"] * X["strength"] + eps)
    X["energy_sin_angle"] = X["energy"] * X["sin_angle"]
    X["momentum_proxy"] = X["effective_energy"] * X["sin_angle"]

    # Risk-controlled atmosphere ratio
    X["coupling_per_atm_clipped"] = X["coupling"] / (X["atmosphere"] + 1e-3)
    X["log_coupling_per_atm"] = np.log1p(X["coupling_per_atm_clipped"])
    X["retention_factor"] = X["atmosphere"] * X["drag_proxy"] / (X["energy"] + eps)

    return X

X_base_train = add_physics_features(raw_train)
X_base_test = add_physics_features(raw_test)

print("X_base_train:", X_base_train.shape)
print("X_base_test:", X_base_test.shape)

X_base_train: (2930, 48)
X_base_test: (492, 48)


## 4. Advanced Physics Features v2

We add only features that are not pure duplicates of the existing ones.

The most relevant are:

- `froude_proxy`: square-root version of effective energy scaled by gravity. This can help distance targets if the relation behaves closer to a velocity-like scaling.
- `stress_ratio_compact`: effective energy divided by compact material resistance. This is a fragmentation and size proxy.
- `aero_braking_index`: equivalent to `drag_per_gravity`; kept as an alias only for diagnostics.
- `escape_ratio`: equivalent to `effective_energy_per_gravity`; kept as an alias only for diagnostics.

For fair modeling, we mainly test `froude_proxy` and `stress_ratio_compact`, while tracking the aliases to confirm redundancy.

In [4]:
def add_advanced_features_to_existing(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    eps = 1e-9

    # Velocity-like / Froude-inspired proxy.
    X["froude_proxy"] = np.sqrt(X["effective_energy"] / (X["gravity"] + eps))

    # Same physical idea as effective_energy_per_gravity, kept for interpretability check.
    X["escape_ratio"] = X["effective_energy"] / (X["gravity"] + eps)

    # Same physical idea as drag_per_gravity, kept for interpretability check.
    X["aero_braking_index"] = (X["atmosphere"] * X["shape_factor"]) / (X["gravity"] + eps)

    # Energy transferred relative to compact material resistance.
    X["stress_ratio_compact"] = X["effective_energy"] / (X["material_resistance_index"] + eps)

    # Optional nonlinear variants that may help tree models only if useful.
    X["sqrt_effective_energy_per_strength"] = np.sqrt(X["effective_energy_per_strength"].clip(lower=0))
    X["sqrt_effective_energy_per_gravity"] = np.sqrt(X["effective_energy_per_gravity"].clip(lower=0))

    return X

X_v2_train = add_advanced_features_to_existing(X_base_train)
X_v2_test = add_advanced_features_to_existing(X_base_test)

new_advanced_features = [
    "froude_proxy",
    "stress_ratio_compact",
    "sqrt_effective_energy_per_strength",
    "sqrt_effective_energy_per_gravity",
]

redundancy_check_features = [
    "escape_ratio",
    "aero_braking_index",
]

print("X_v2_train:", X_v2_train.shape)
print("X_v2_test:", X_v2_test.shape)
print("New advanced features:", new_advanced_features)
print("Redundancy check features:", redundancy_check_features)

display(X_v2_train[new_advanced_features + redundancy_check_features].describe().T)

X_v2_train: (2930, 54)
X_v2_test: (492, 54)
New advanced features: ['froude_proxy', 'stress_ratio_compact', 'sqrt_effective_energy_per_strength', 'sqrt_effective_energy_per_gravity']
Redundancy check features: ['escape_ratio', 'aero_braking_index']


,count,mean,std,min,25%,50%,75%,max
froude_proxy,2930.0,0.861504,0.357470,0.334587,0.594007,0.763417,1.092141,2.077954
stress_ratio_compact,2930.0,1.738920,1.500413,0.360757,0.576322,0.751499,2.818246,9.036020
sqrt_effective_energy_per_strength,2930.0,1.064822,0.396246,0.593004,0.742567,0.841218,1.396644,2.466712
sqrt_effective_energy_per_gravity,2930.0,0.861504,0.357470,0.334587,0.594007,0.763417,1.092141,2.077954
escape_ratio,2930.0,0.869930,0.731380,0.111949,0.352845,0.582806,1.192771,4.317892
aero_braking_index,2930.0,0.124215,0.142810,0.004448,0.028599,0.065925,0.164735,0.692683


## 5. Feature Sets

We rebuild the same feature sets as the final model and then add v2 variants.

In [5]:
raw_features = input_cols.copy()

core_physics_features = raw_features + [
    "effective_energy", "log_energy", "log_effective_energy",
    "sin_angle", "cos_angle", "tan_angle",
    "horizontal_energy", "vertical_energy", "vertical_horizontal_ratio",
    "energy_per_strength", "effective_energy_per_strength",
    "material_resistance_index", "fragmentation_index",
    "coupling_porosity", "coupling_atmosphere", "porosity_strength",
    "energy_per_gravity", "effective_energy_per_gravity",
    "horizontal_energy_per_gravity", "vertical_energy_per_gravity",
    "drag_proxy", "drag_per_gravity", "atmosphere_shape_energy", "atmosphere_per_gravity",
]

extended_physics_features = core_physics_features + [
    "pi_gravity_proxy", "pi_strength_proxy", "pi_atmosphere_proxy",
    "scaled_energy", "fragility", "range_proxy",
    "energy_sin_angle", "momentum_proxy",
    "coupling_per_atm_clipped", "log_coupling_per_atm", "retention_factor",
]

extended_plus_regimes_features = extended_physics_features + [
    "porosity_regime", "strength_regime", "angle_regime", "atm_regime", "regime_combo"
]

fragmentation_features = raw_features + [
    "effective_energy", "log_effective_energy",
    "sin_angle", "cos_angle", "horizontal_energy", "vertical_energy",
    "energy_per_strength", "effective_energy_per_strength",
    "material_resistance_index", "fragmentation_index",
    "coupling_porosity", "coupling_atmosphere", "porosity_strength",
    "drag_proxy", "atmosphere_shape_energy",
    "pi_strength_proxy", "pi_atmosphere_proxy", "scaled_energy", "fragility",
    "energy_sin_angle", "momentum_proxy",
    "porosity_regime", "strength_regime", "angle_regime", "atm_regime", "regime_combo",
]

distance_features = raw_features + [
    "effective_energy", "log_effective_energy",
    "sin_angle", "cos_angle", "horizontal_energy", "vertical_energy",
    "energy_per_gravity", "effective_energy_per_gravity",
    "horizontal_energy_per_gravity", "vertical_energy_per_gravity",
    "drag_proxy", "drag_per_gravity", "atmosphere_shape_energy", "atmosphere_per_gravity",
    "pi_gravity_proxy", "pi_atmosphere_proxy", "scaled_energy", "range_proxy",
    "energy_sin_angle", "momentum_proxy",
    "porosity_regime", "strength_regime", "angle_regime", "atm_regime", "regime_combo",
]

def clean_feature_list(features, X):
    seen = set()
    out = []
    for f in features:
        if f in X.columns and f not in seen:
            out.append(f)
            seen.add(f)
    return out

raw_features = clean_feature_list(raw_features, X_v2_train)
core_physics_features = clean_feature_list(core_physics_features, X_v2_train)
extended_physics_features = clean_feature_list(extended_physics_features, X_v2_train)
extended_plus_regimes_features = clean_feature_list(extended_plus_regimes_features, X_v2_train)
fragmentation_features = clean_feature_list(fragmentation_features, X_v2_train)
distance_features = clean_feature_list(distance_features, X_v2_train)

fragmentation_features_v2 = clean_feature_list(fragmentation_features + [
    "stress_ratio_compact",
    "sqrt_effective_energy_per_strength",
], X_v2_train)

distance_features_v2 = clean_feature_list(distance_features + [
    "froude_proxy",
    "sqrt_effective_energy_per_gravity",
], X_v2_train)

extended_plus_regimes_features_v2 = clean_feature_list(extended_plus_regimes_features + new_advanced_features, X_v2_train)

feature_sets = {
    "v1_target_family": None,
    "v2_target_family": None,
    "v1_extended_plus_regimes": extended_plus_regimes_features,
    "v2_extended_plus_regimes": extended_plus_regimes_features_v2,
}

def get_features_for_version(feature_set_name: str, target: str):
    if feature_set_name == "v1_target_family":
        return fragmentation_features if target in fragmentation_targets else distance_features
    if feature_set_name == "v2_target_family":
        return fragmentation_features_v2 if target in fragmentation_targets else distance_features_v2
    return feature_sets[feature_set_name]

summary = []
for name in feature_sets:
    if name.endswith("target_family"):
        summary.append({"feature_set": name, "n_features": "target-dependent"})
    else:
        summary.append({"feature_set": name, "n_features": len(feature_sets[name])})
summary.append({"feature_set": "fragmentation_v1", "n_features": len(fragmentation_features)})
summary.append({"feature_set": "fragmentation_v2", "n_features": len(fragmentation_features_v2)})
summary.append({"feature_set": "distance_v1", "n_features": len(distance_features)})
summary.append({"feature_set": "distance_v2", "n_features": len(distance_features_v2)})

display(pd.DataFrame(summary))

,feature_set,n_features
0,v1_target_family,target-dependent
1,v2_target_family,target-dependent
2,v1_extended_plus_regimes,48
3,v2_extended_plus_regimes,52
4,fragmentation_v1,34
5,fragmentation_v2,36
6,distance_v1,33
7,distance_v2,35


## 6. Metrics and Constraint Diagnostics

In [6]:
def regression_metrics(y_true, y_pred, target_name=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    out = {"MAE": mae, "RMSE": rmse, "R2": r2}
    if target_name is not None:
        target_std = y[target_name].std()
        out["normalized_MAE"] = mae / (target_std + 1e-9)
        out["normalized_RMSE"] = rmse / (target_std + 1e-9)
    return out


def error_distribution_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    abs_error = np.abs(y_pred - y_true)
    residual = y_pred - y_true
    return {
        "Median_AE": np.median(abs_error),
        "P90_AE": np.percentile(abs_error, 90),
        "P95_AE": np.percentile(abs_error, 95),
        "Max_AE": np.max(abs_error),
        "Bias": np.mean(residual),
    }


def clip_predictions(preds, target):
    preds = np.asarray(preds).copy()
    if target in ["fines_frac", "oversize_frac"]:
        return np.clip(preds, 0, 1)
    return np.clip(preds, 0, None)


def feasibility_mask(df_targets):
    return (df_targets["P80"].between(p80_min, p80_max)) & (df_targets["R95"] <= r95_max)


def near_feasible_mask(df_targets):
    return (df_targets["P80"].between(80, 120)) & (df_targets["R95"] <= 250)


def constraint_violation_score(df_pred):
    p80_low = np.maximum(p80_min - df_pred["P80"], 0)
    p80_high = np.maximum(df_pred["P80"] - p80_max, 0)
    r95_violation = np.maximum(df_pred["R95"] - r95_max, 0)
    return p80_low + p80_high + r95_violation


def custom_constraint_metrics(y_true_df, y_pred_df):
    true_feasible = feasibility_mask(y_true_df)
    pred_feasible = feasibility_mask(y_pred_df)
    near_zone = near_feasible_mask(y_true_df)

    tp = int((true_feasible & pred_feasible).sum())
    fp = int((~true_feasible & pred_feasible).sum())
    fn = int((true_feasible & ~pred_feasible).sum())

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    violation = constraint_violation_score(y_pred_df)

    metrics = {
        "n_true_feasible": int(true_feasible.sum()),
        "n_pred_feasible": int(pred_feasible.sum()),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "feasible_precision": precision,
        "feasible_recall": recall,
        "feasible_f1": f1,
        "mean_pred_constraint_violation": float(violation.mean()),
        "median_pred_constraint_violation": float(np.median(violation)),
        "near_zone_count": int(near_zone.sum()),
    }

    if near_zone.sum() > 0:
        metrics["near_zone_MAE_P80"] = mean_absolute_error(
            y_true_df.loc[near_zone, "P80"], y_pred_df.loc[near_zone, "P80"]
        )
        metrics["near_zone_MAE_R95"] = mean_absolute_error(
            y_true_df.loc[near_zone, "R95"], y_pred_df.loc[near_zone, "R95"]
        )
        metrics["near_zone_R95_bias"] = float((
            y_pred_df.loc[near_zone, "R95"] - y_true_df.loc[near_zone, "R95"]
        ).mean())

    return metrics

## 7. Model Builders

We keep the same robust ExtraTrees configuration as the current final model.
For the v2 experiment, we mainly test ExtraTrees direct and ExtraTrees with a log target transformation for distances.

In [7]:
def build_extratrees(random_state=42, n_estimators=800):
    return ExtraTreesRegressor(
        n_estimators=n_estimators,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=random_state,
        n_jobs=-1,
    )


def build_model_for_strategy(strategy_name, target, random_state=42):
    base = build_extratrees(random_state=random_state, n_estimators=800)

    if strategy_name.endswith("log") and target in distance_targets:
        return TransformedTargetRegressor(
            regressor=base,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    return base

## 8. Cross-Validation Functions

This section compares v1 and v2 fairly using the same KFold splits.

In [8]:
def cross_validate_target_strategy(
    X, y_df, target, features, strategy_name,
    n_splits=5, random_state=42
):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(y_df))
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
        X_train = X.iloc[train_idx][features]
        X_valid = X.iloc[valid_idx][features]
        y_train = y_df.iloc[train_idx][target]
        y_valid = y_df.iloc[valid_idx][target]

        model = build_model_for_strategy(strategy_name, target, random_state=random_state + fold)
        model.fit(X_train, y_train)

        pred = model.predict(X_valid)
        pred = clip_predictions(pred, target)
        oof[valid_idx] = pred

        metrics = regression_metrics(y_valid, pred, target_name=target)
        metrics.update(error_distribution_metrics(y_valid, pred))
        metrics.update({"fold": fold, "target": target, "strategy": strategy_name, "n_features": len(features)})
        fold_rows.append(metrics)

    overall = regression_metrics(y_df[target], oof, target_name=target)
    overall.update(error_distribution_metrics(y_df[target], oof))
    overall.update({"fold": "OOF", "target": target, "strategy": strategy_name, "n_features": len(features)})

    return pd.DataFrame(fold_rows), overall, oof


def evaluate_strategy(strategy_name, X, y_df, feature_set_name, n_splits=5):
    overall_rows = []
    fold_rows = []
    oof_df = pd.DataFrame(index=y_df.index)

    for target in target_cols:
        features = get_features_for_version(feature_set_name, target)
        fold_df, overall, oof = cross_validate_target_strategy(
            X=X,
            y_df=y_df,
            target=target,
            features=features,
            strategy_name=strategy_name,
            n_splits=n_splits,
        )
        fold_df["feature_set"] = feature_set_name
        overall["feature_set"] = feature_set_name

        fold_rows.append(fold_df)
        overall_rows.append(overall)
        oof_df[target] = oof

    overall_df = pd.DataFrame(overall_rows)
    folds_df = pd.concat(fold_rows, ignore_index=True)
    constraint_df = pd.DataFrame([custom_constraint_metrics(y_df, oof_df)])
    constraint_df["strategy"] = strategy_name
    constraint_df["feature_set"] = feature_set_name

    return overall_df, folds_df, oof_df, constraint_df

## 9. Evaluate v1 vs v2 Strategies

Strategies tested:

- `v1_direct`: current feature design, direct target regression.
- `v2_direct`: current features + advanced features, direct target regression.
- `v2_log`: current features + advanced features, log transform for distance targets only.

This can take several minutes.

In [9]:
RUN_FULL_EVALUATION = True

all_overall = []
all_folds = []
all_constraints = []
oof_store = {}

if RUN_FULL_EVALUATION:
    strategies = [
        ("v1_direct", X_base_train, "v1_target_family"),
        ("v2_direct", X_v2_train, "v2_target_family"),
        ("v2_log", X_v2_train, "v2_target_family"),
    ]

    for strategy_name, X_matrix, feature_set_name in strategies:
        print(f"Evaluating {strategy_name} with {feature_set_name}...")
        overall_df, folds_df, oof_df, constraint_df = evaluate_strategy(
            strategy_name=strategy_name,
            X=X_matrix,
            y_df=y,
            feature_set_name=feature_set_name,
            n_splits=5,
        )
        all_overall.append(overall_df)
        all_folds.append(folds_df)
        all_constraints.append(constraint_df)
        oof_store[strategy_name] = oof_df

    results_df = pd.concat(all_overall, ignore_index=True)
    folds_results_df = pd.concat(all_folds, ignore_index=True)
    constraint_results_df = pd.concat(all_constraints, ignore_index=True)

    display(results_df.sort_values(["target", "normalized_MAE"]))
    display(constraint_results_df.sort_values("feasible_f1", ascending=False))
else:
    print("Set RUN_FULL_EVALUATION=True to run the comparison.")

Evaluating v1_direct with v1_target_family...
Evaluating v2_direct with v2_target_family...
Evaluating v2_log with v2_target_family...


,MAE,RMSE,R2,normalized_MAE,normalized_RMSE,Median_AE,P90_AE,P95_AE,Max_AE,Bias,fold,target,strategy,n_features,feature_set
6,7.649464,10.159581,0.976120,0.116332,0.154506,5.889286,16.804011,21.487673,53.412408,-0.005756,OOF,P80,v2_direct,36,v2_target_family
12,7.649464,10.159581,0.976120,0.116332,0.154506,5.889286,16.804011,21.487673,53.412408,-0.005756,OOF,P80,v2_log,36,v2_target_family
0,7.711179,10.257217,0.975659,0.117271,0.155991,5.946852,16.858204,21.748532,54.213116,-0.008502,OOF,P80,v1_direct,34,v1_target_family
16,50.209967,79.025966,0.897517,0.203363,0.320074,26.846766,128.166791,174.316223,546.542138,-6.754301,OOF,R50_fines,v2_log,35,v2_target_family
4,50.329516,78.446782,0.899014,0.203847,0.317729,26.780948,129.784281,171.747276,532.805439,-0.407495,OOF,R50_fines,v1_direct,33,v1_target_family
10,50.497975,78.643481,0.898507,0.204529,0.318525,27.667037,130.263262,173.637582,537.300844,-0.398513,OOF,R50_fines,v2_direct,35,v2_target_family
5,22.721199,38.564809,0.873475,0.209534,0.355643,11.681322,57.277830,81.805393,349.482922,-0.002926,OOF,R50_oversize,v1_direct,33,v1_target_family
17,22.758461,39.022716,0.870453,0.209877,0.359865,11.458278,56.295026,83.132699,356.305980,-3.384579,OOF,R50_oversize,v2_log,35,v2_target_family
11,22.765811,38.664968,0.872817,0.209945,0.356566,11.713865,57.147167,81.656886,346.399054,-0.006308,OOF,R50_oversize,v2_direct,35,v2_target_family
3,41.404424,68.491023,0.917533,0.173572,0.287122,20.724224,103.213177,151.901190,469.786212,-0.437139,OOF,R95,v1_direct,33,v1_target_family


,n_true_feasible,n_pred_feasible,true_positive,false_positive,false_negative,feasible_precision,feasible_recall,feasible_f1,mean_pred_constraint_violation,median_pred_constraint_violation,near_zone_count,near_zone_MAE_P80,near_zone_MAE_R95,near_zone_R95_bias,strategy,feature_set
1,35,37,13,24,22,0.351351,0.371429,0.361111,199.973001,135.976335,372,4.584673,25.643486,13.820069,v2_direct,v2_target_family
2,35,37,13,24,22,0.351351,0.371429,0.361111,194.808062,135.203469,372,4.584673,23.954674,9.841835,v2_log,v2_target_family
0,35,39,13,26,22,0.333333,0.371429,0.351351,199.695516,135.196588,372,4.571787,25.516818,13.959853,v1_direct,v1_target_family


## 10. Target-Level Comparison

This section identifies whether v2 improves each target. We look at both MAE and RMSE/P95 because the challenge may penalize large errors more than average errors.

In [10]:
if RUN_FULL_EVALUATION:
    metric_cols = [
        "MAE", "RMSE", "R2", "normalized_MAE", "normalized_RMSE",
        "Median_AE", "P90_AE", "P95_AE", "Max_AE", "Bias"
    ]

    comparison = results_df.pivot_table(
        index="target",
        columns="strategy",
        values=["MAE", "RMSE", "R2", "normalized_MAE", "normalized_RMSE", "P95_AE"],
        aggfunc="first",
    )

    display(comparison)

    best_by_mae = results_df.sort_values(["target", "MAE"]).groupby("target", as_index=False).first()
    best_by_rmse = results_df.sort_values(["target", "RMSE"]).groupby("target", as_index=False).first()
    best_by_p95 = results_df.sort_values(["target", "P95_AE"]).groupby("target", as_index=False).first()

    print("Best by MAE")
    display(best_by_mae[["target", "strategy", "feature_set", "MAE", "RMSE", "R2", "P95_AE", "n_features"]])

    print("Best by RMSE")
    display(best_by_rmse[["target", "strategy", "feature_set", "MAE", "RMSE", "R2", "P95_AE", "n_features"]])

    print("Best by P95_AE")
    display(best_by_p95[["target", "strategy", "feature_set", "MAE", "RMSE", "R2", "P95_AE", "n_features"]])

MAE                            P95_AE                                R2                           RMSE                       normalized_MAE            \
strategy       v1_direct  v2_direct     v2_log   v1_direct   v2_direct      v2_log v1_direct v2_direct    v2_log  v1_direct  v2_direct     v2_log      v1_direct v2_direct   
target                                                                                                                                                                       
P80             7.711179   7.649464   7.649464   21.748532   21.487673   21.487673  0.975659  0.976120  0.976120  10.257217  10.159581  10.159581       0.117271  0.116332   
R50_fines      50.329516  50.497975  50.209967  171.747276  173.637582  174.316223  0.899014  0.898507  0.897517  78.446782  78.643481  79.025966       0.203847  0.204529   
R50_oversize   22.721199  22.765811  22.758461   81.805393   81.656886   83.132699  0.873475  0.872817  0.870453  38.564809  38.664968  39.022716       0.209534  0.209945   
R95            41.404424  41.697002  41.701015  151.901190  150.372082  152.864078  0.917533  0.916213  0.913653  68.491023  69.036775  70.083571       0.173572  0.174799   
fines_frac      0.006487   0.006312   0.006312    0.031649    0.031030    0.031030  0.956900  0.959491  0.959491   0.014240   0.013805   0.013805       0.094558  0.092001   
oversize_frac   0.026200   0.025643   0.025643    0.075833    0.074408    0.074408  0.990204  0.990578  0.990578   0.035832   0.035141   0.035141       0.072357  0.070821   

                        normalized_RMSE                      
strategy         v2_log       v1_direct v2_direct    v2_log  
target                                                       
P80            0.116332        0.155991  0.154506  0.154506  
R50_fines      0.203363        0.317729  0.318525  0.320074  
R50_oversize   0.209877        0.355643  0.356566  0.359865  
R95            0.174815        0.287122  0.289410  0.293798  
fines_frac     0.092001        0.207570  0.201233  0.201233  
oversize_frac  0.070821        0.098959  0.097050  0.097050

Best by MAE


,target,strategy,feature_set,MAE,RMSE,R2,P95_AE,n_features
0,P80,v2_direct,v2_target_family,7.649464,10.159581,0.976120,21.487673,36
1,R50_fines,v2_log,v2_target_family,50.209967,79.025966,0.897517,174.316223,35
2,R50_oversize,v1_direct,v1_target_family,22.721199,38.564809,0.873475,81.805393,33
3,R95,v1_direct,v1_target_family,41.404424,68.491023,0.917533,151.901190,33
4,fines_frac,v2_direct,v2_target_family,0.006312,0.013805,0.959491,0.031030,36
5,oversize_frac,v2_direct,v2_target_family,0.025643,0.035141,0.990578,0.074408,36


Best by RMSE


,target,strategy,feature_set,MAE,RMSE,R2,P95_AE,n_features
0,P80,v2_log,v2_target_family,7.649464,10.159581,0.976120,21.487673,36
1,R50_fines,v1_direct,v1_target_family,50.329516,78.446782,0.899014,171.747276,33
2,R50_oversize,v1_direct,v1_target_family,22.721199,38.564809,0.873475,81.805393,33
3,R95,v1_direct,v1_target_family,41.404424,68.491023,0.917533,151.901190,33
4,fines_frac,v2_direct,v2_target_family,0.006312,0.013805,0.959491,0.031030,36
5,oversize_frac,v2_direct,v2_target_family,0.025643,0.035141,0.990578,0.074408,36


Best by P95_AE


,target,strategy,feature_set,MAE,RMSE,R2,P95_AE,n_features
0,P80,v2_direct,v2_target_family,7.649464,10.159581,0.976120,21.487673,36
1,R50_fines,v1_direct,v1_target_family,50.329516,78.446782,0.899014,171.747276,33
2,R50_oversize,v2_direct,v2_target_family,22.765811,38.664968,0.872817,81.656886,35
3,R95,v2_direct,v2_target_family,41.697002,69.036775,0.916213,150.372082,35
4,fines_frac,v2_direct,v2_target_family,0.006312,0.013805,0.959491,0.031030,36
5,oversize_frac,v2_direct,v2_target_family,0.025643,0.035141,0.990578,0.074408,36


## 11. Improvement Table vs v1

Positive percentages mean v2 improved relative to v1.

In [11]:
if RUN_FULL_EVALUATION:
    base = results_df[results_df["strategy"] == "v1_direct"].set_index("target")
    rows = []

    for strategy in ["v2_direct", "v2_log"]:
        curr = results_df[results_df["strategy"] == strategy].set_index("target")
        for target in target_cols:
            row = {
                "target": target,
                "strategy": strategy,
                "MAE_base": base.loc[target, "MAE"],
                "MAE_current": curr.loc[target, "MAE"],
                "MAE_improvement_pct": 100 * (base.loc[target, "MAE"] - curr.loc[target, "MAE"]) / base.loc[target, "MAE"],
                "RMSE_base": base.loc[target, "RMSE"],
                "RMSE_current": curr.loc[target, "RMSE"],
                "RMSE_improvement_pct": 100 * (base.loc[target, "RMSE"] - curr.loc[target, "RMSE"]) / base.loc[target, "RMSE"],
                "P95_base": base.loc[target, "P95_AE"],
                "P95_current": curr.loc[target, "P95_AE"],
                "P95_improvement_pct": 100 * (base.loc[target, "P95_AE"] - curr.loc[target, "P95_AE"]) / base.loc[target, "P95_AE"],
            }
            rows.append(row)

    improvement_df = pd.DataFrame(rows)
    display(improvement_df.sort_values(["target", "strategy"]))

,target,strategy,MAE_base,MAE_current,MAE_improvement_pct,RMSE_base,RMSE_current,RMSE_improvement_pct,P95_base,P95_current,P95_improvement_pct
0,P80,v2_direct,7.711179,7.649464,0.800334,10.257217,10.159581,0.951873,21.748532,21.487673,1.199433
6,P80,v2_log,7.711179,7.649464,0.800334,10.257217,10.159581,0.951873,21.748532,21.487673,1.199433
4,R50_fines,v2_direct,50.329516,50.497975,-0.334712,78.446782,78.643481,-0.250742,171.747276,173.637582,-1.100633
10,R50_fines,v2_log,50.329516,50.209967,0.237532,78.446782,79.025966,-0.738315,171.747276,174.316223,-1.495772
5,R50_oversize,v2_direct,22.721199,22.765811,-0.196342,38.564809,38.664968,-0.259716,81.805393,81.656886,0.181537
11,R50_oversize,v2_log,22.721199,22.758461,-0.163996,38.564809,39.022716,-1.187372,81.805393,83.132699,-1.622517
3,R95,v2_direct,41.404424,41.697002,-0.706635,68.491023,69.036775,-0.796823,151.901190,150.372082,1.006647
9,R95,v2_log,41.404424,41.701015,-0.716327,68.491023,70.083571,-2.325194,151.901190,152.864078,-0.633891
1,fines_frac,v2_direct,0.006487,0.006312,2.704118,0.014240,0.013805,3.052734,0.031649,0.031030,1.954334
7,fines_frac,v2_log,0.006487,0.006312,2.704118,0.014240,0.013805,3.052734,0.031649,0.031030,1.954334


## 12. Constraint and Near-Zone Comparison

For this challenge, global metrics are not enough. We also compare how strategies behave near the inverse-design zone.

In [12]:
if RUN_FULL_EVALUATION:
    display(constraint_results_df[[
        "strategy", "feature_set", "n_true_feasible", "n_pred_feasible",
        "true_positive", "false_positive", "false_negative",
        "feasible_precision", "feasible_recall", "feasible_f1",
        "near_zone_count", "near_zone_MAE_P80", "near_zone_MAE_R95", "near_zone_R95_bias",
        "mean_pred_constraint_violation", "median_pred_constraint_violation",
    ]].sort_values("feasible_f1", ascending=False))

,strategy,feature_set,n_true_feasible,n_pred_feasible,true_positive,false_positive,false_negative,feasible_precision,feasible_recall,feasible_f1,near_zone_count,near_zone_MAE_P80,near_zone_MAE_R95,near_zone_R95_bias,mean_pred_constraint_violation,median_pred_constraint_violation
1,v2_direct,v2_target_family,35,37,13,24,22,0.351351,0.371429,0.361111,372,4.584673,25.643486,13.820069,199.973001,135.976335
2,v2_log,v2_target_family,35,37,13,24,22,0.351351,0.371429,0.361111,372,4.584673,23.954674,9.841835,194.808062,135.203469
0,v1_direct,v1_target_family,35,39,13,26,22,0.333333,0.371429,0.351351,372,4.571787,25.516818,13.959853,199.695516,135.196588


## 13. Error Analysis by Gravity Regime

Distance targets are strongly controlled by gravity. This block checks whether v2 helps or hurts by gravity regime.

In [13]:
if RUN_FULL_EVALUATION:
    gravity_rows = []
    for strategy_name, oof_df in oof_store.items():
        for target in distance_targets:
            temp = pd.DataFrame({
                "gravity": raw_train["gravity"],
                "true": y[target],
                "pred": oof_df[target],
            })
            temp["abs_error"] = np.abs(temp["pred"] - temp["true"])
            grouped = temp.groupby("gravity").agg(
                n=("abs_error", "size"),
                MAE=("abs_error", "mean"),
                Median_AE=("abs_error", "median"),
                true_mean=("true", "mean"),
                pred_mean=("pred", "mean"),
            ).reset_index()
            grouped["relative_MAE"] = grouped["MAE"] / (grouped["true_mean"].abs() + 1e-9)
            grouped["strategy"] = strategy_name
            grouped["target"] = target
            gravity_rows.append(grouped)

    gravity_error_df = pd.concat(gravity_rows, ignore_index=True)
    display(gravity_error_df.sort_values(["target", "gravity", "relative_MAE"]))

    for target in distance_targets:
        pivot = gravity_error_df[gravity_error_df["target"] == target].pivot_table(
            index="gravity", columns="strategy", values="relative_MAE", aggfunc="first"
        )
        print(f"Relative MAE by gravity - {target}")
        display(pivot)

,gravity,n,MAE,Median_AE,true_mean,pred_mean,relative_MAE,strategy,target
21,1.62,990,99.020811,82.579894,617.765891,603.669671,0.160289,v2_log,R50_fines
3,1.62,990,99.293323,83.633729,617.765891,615.801966,0.160730,v1_direct,R50_fines
12,1.62,990,99.603792,83.848829,617.765891,615.917262,0.161232,v2_direct,R50_fines
22,3.71,1000,37.288061,31.044509,231.088286,226.693290,0.161359,v2_log,R50_fines
4,3.71,1000,37.362100,31.447645,231.088286,231.610999,0.161679,v1_direct,R50_fines
13,3.71,1000,37.550298,30.984090,231.088286,231.598623,0.162493,v2_direct,R50_fines
23,9.81,940,12.549510,10.008128,72.569602,71.037849,0.172931,v2_log,R50_fines
14,9.81,940,12.554270,10.381746,72.569602,72.731476,0.172996,v2_direct,R50_fines
5,9.81,940,12.556375,10.302607,72.569602,72.811744,0.173025,v1_direct,R50_fines
6,1.62,990,44.996866,34.360845,251.674665,251.249420,0.178790,v1_direct,R50_oversize


Relative MAE by gravity - R95


strategy,v1_direct,v2_direct,v2_log
gravity,,,
1.62,0.161969,0.163309,0.163761
3.71,0.164563,0.165362,0.164407
9.81,0.173358,0.174051,0.173426


Relative MAE by gravity - R50_fines


strategy,v1_direct,v2_direct,v2_log
gravity,,,
1.62,0.160730,0.161232,0.160289
3.71,0.161679,0.162493,0.161359
9.81,0.173025,0.172996,0.172931


Relative MAE by gravity - R50_oversize


strategy,v1_direct,v2_direct,v2_log
gravity,,,
1.62,0.178790,0.179131,0.179766
3.71,0.179711,0.180014,0.178179
9.81,0.192717,0.193346,0.193010


## 14. Feature Redundancy Diagnostics

This checks whether the new aliases are redundant with existing features.
High correlation does not automatically mean a feature is harmful for tree models, but it tells us interpretation may be duplicated.

In [14]:
redundancy_pairs = [
    ("escape_ratio", "effective_energy_per_gravity"),
    ("aero_braking_index", "drag_per_gravity"),
    ("froude_proxy", "sqrt_effective_energy_per_gravity"),
    ("stress_ratio_compact", "effective_energy_per_strength"),
    ("stress_ratio_compact", "fragmentation_index"),
]

rows = []
for a, b in redundancy_pairs:
    if a in X_v2_train.columns and b in X_v2_train.columns:
        rows.append({
            "feature_a": a,
            "feature_b": b,
            "pearson_corr": X_v2_train[a].corr(X_v2_train[b], method="pearson"),
            "spearman_corr": X_v2_train[a].corr(X_v2_train[b], method="spearman"),
        })

display(pd.DataFrame(rows))

,feature_a,feature_b,pearson_corr,spearman_corr
0,escape_ratio,effective_energy_per_gravity,1.000000,1.000000
1,aero_braking_index,drag_per_gravity,1.000000,1.000000
2,froude_proxy,sqrt_effective_energy_per_gravity,1.000000,1.000000
3,stress_ratio_compact,effective_energy_per_strength,0.996986,0.996132
4,stress_ratio_compact,fragmentation_index,0.993930,0.911147


## 15. Decision Rule

We decide whether to upgrade from v1 to v2 using a conservative rule:

- Keep v1 if gains are tiny or inconsistent.
- Upgrade a target to v2 only if MAE improves and RMSE or P95 does not get worse materially.
- Be extra careful with `v2_log`: if it improves MAE but worsens RMSE/P95, it may not be safer for a competition.

In [15]:
def decide_target_strategy(results_df, tolerance_pct=0.5):
    """
    Select strategy per target.
    Rule:
    - Start from v1_direct.
    - Prefer a v2 strategy only if MAE improves by at least tolerance_pct
      and RMSE does not degrade by more than tolerance_pct.
    - If P95 improves strongly, allow small MAE tie.
    """
    decisions = []
    base = results_df[results_df["strategy"] == "v1_direct"].set_index("target")

    for target in target_cols:
        base_row = base.loc[target]
        candidates = results_df[results_df["target"] == target].copy()
        best_choice = "v1_direct"
        reason = "keep v1 by default"

        for _, row in candidates[candidates["strategy"] != "v1_direct"].iterrows():
            mae_impr = 100 * (base_row["MAE"] - row["MAE"]) / base_row["MAE"]
            rmse_impr = 100 * (base_row["RMSE"] - row["RMSE"]) / base_row["RMSE"]
            p95_impr = 100 * (base_row["P95_AE"] - row["P95_AE"]) / base_row["P95_AE"]

            if (mae_impr >= tolerance_pct) and (rmse_impr >= -tolerance_pct):
                if best_choice == "v1_direct" or row["MAE"] < candidates[candidates["strategy"] == best_choice]["MAE"].iloc[0]:
                    best_choice = row["strategy"]
                    reason = f"MAE improves {mae_impr:.2f}%, RMSE change {rmse_impr:.2f}%, P95 change {p95_impr:.2f}%"
            elif (p95_impr >= 2.0) and (mae_impr >= -tolerance_pct) and (rmse_impr >= -tolerance_pct):
                if best_choice == "v1_direct":
                    best_choice = row["strategy"]
                    reason = f"P95 improves {p95_impr:.2f}% with acceptable MAE/RMSE changes"

        selected_row = candidates[candidates["strategy"] == best_choice].iloc[0]
        decisions.append({
            "target": target,
            "selected_strategy": best_choice,
            "selected_feature_set": selected_row["feature_set"],
            "MAE": selected_row["MAE"],
            "RMSE": selected_row["RMSE"],
            "R2": selected_row["R2"],
            "P95_AE": selected_row["P95_AE"],
            "reason": reason,
        })

    return pd.DataFrame(decisions)

if RUN_FULL_EVALUATION:
    decision_df = decide_target_strategy(results_df, tolerance_pct=0.5)
    display(decision_df)

    n_upgrades = (decision_df["selected_strategy"] != "v1_direct").sum()
    print("Number of target upgrades suggested:", n_upgrades)

,target,selected_strategy,selected_feature_set,MAE,RMSE,R2,P95_AE,reason
0,P80,v2_direct,v2_target_family,7.649464,10.159581,0.976120,21.487673,"MAE improves 0.80%, RMSE change 0.95%, P95 cha..."
1,fines_frac,v2_direct,v2_target_family,0.006312,0.013805,0.959491,0.031030,"MAE improves 2.70%, RMSE change 3.05%, P95 cha..."
2,oversize_frac,v2_direct,v2_target_family,0.025643,0.035141,0.990578,0.074408,"MAE improves 2.12%, RMSE change 1.93%, P95 cha..."
3,R95,v1_direct,v1_target_family,41.404424,68.491023,0.917533,151.901190,keep v1 by default
4,R50_fines,v1_direct,v1_target_family,50.329516,78.446782,0.899014,171.747276,keep v1 by default
5,R50_oversize,v1_direct,v1_target_family,22.721199,38.564809,0.873475,81.805393,keep v1 by default


Number of target upgrades suggested: 3


## 16. Optional: Train and Save a v2 Final Model if Useful

Run this only if the decision table suggests meaningful upgrades.

If no meaningful upgrades are found, keep the existing `05_final_forward_model.ipynb` output as the final forward model v1.

In [18]:
class TargetSpecificForwardModelV2:
    def __init__(self, target_decisions, random_state=42):
        self.target_decisions = target_decisions
        self.random_state = random_state
        self.models_ = {}

    def fit(self, raw_X, y_df):
        X_base = add_physics_features(raw_X)
        X_v2 = add_advanced_features_to_existing(X_base)

        for i, row in self.target_decisions.iterrows():
            target = row["target"]
            strategy = row["selected_strategy"]
            feature_set = row["selected_feature_set"]

            if strategy == "v1_direct":
                X_matrix = X_base
            else:
                X_matrix = X_v2

            features = get_features_for_version(feature_set, target)
            model = build_model_for_strategy(strategy, target, random_state=self.random_state + i)
            model.fit(X_matrix[features], y_df[target])

            self.models_[target] = {
                "model": model,
                "strategy": strategy,
                "feature_set": feature_set,
                "features": features,
            }

        return self

    def predict(self, raw_X):
        X_base = add_physics_features(raw_X)
        X_v2 = add_advanced_features_to_existing(X_base)
        preds = pd.DataFrame(index=raw_X.index)

        for target, info in self.models_.items():
            X_matrix = X_base if info["strategy"] == "v1_direct" else X_v2
            pred = info["model"].predict(X_matrix[info["features"]])
            preds[target] = clip_predictions(pred, target)

        return preds[target_cols]

    def describe(self):
        return pd.DataFrame([
            {
                "target": target,
                "strategy": info["strategy"],
                "feature_set": info["feature_set"],
                "n_features": len(info["features"]),
            }
            for target, info in self.models_.items()
        ])


TRAIN_AND_SAVE_V2 = True

if TRAIN_AND_SAVE_V2:
    final_v2_model = TargetSpecificForwardModelV2(decision_df, random_state=42)
    final_v2_model.fit(raw_train, y)
    display(final_v2_model.describe())

    test_predictions_v2 = final_v2_model.predict(raw_test)
    display(test_predictions_v2.describe().T)

    submission_v2 = pd.DataFrame({"scenario_id": np.arange(len(raw_test))})
    for col in target_cols:
        submission_v2[col] = test_predictions_v2[col].values

    submission_v2 = submission_v2[[
        "scenario_id", "P80", "fines_frac", "oversize_frac",
        "R95", "R50_fines", "R50_oversize"
    ]]

    submission_v2_path = SUBMISSIONS_DIR / "prediction_submission_forward_v2_advanced_features.csv"
    submission_v2.to_csv(submission_v2_path, index=False)

    model_v2_path = MODELS_DIR / "final_forward_v2_advanced_features.joblib"
    joblib.dump(final_v2_model, model_v2_path)

    print("Saved v2 submission:", submission_v2_path)
    print("Saved v2 model:", model_v2_path)
else:
    print("TRAIN_AND_SAVE_V2=False. Review the decision table before saving a v2 model.")

,target,strategy,feature_set,n_features
0,P80,v2_direct,v2_target_family,36
1,fines_frac,v2_direct,v2_target_family,36
2,oversize_frac,v2_direct,v2_target_family,36
3,R95,v1_direct,v1_target_family,33
4,R50_fines,v1_direct,v1_target_family,33
5,R50_oversize,v1_direct,v1_target_family,33


,count,mean,std,min,25%,50%,75%,max
P80,492.0,157.708072,33.479974,76.016131,130.253486,153.137936,182.044124,230.519576
fines_frac,492.0,0.050271,0.059135,0.001579,0.005592,0.017260,0.085643,0.265903
oversize_frac,492.0,0.468492,0.210192,0.033226,0.289992,0.437667,0.648656,0.847572
R95,492.0,298.530443,241.158254,48.776910,109.611058,177.453510,508.553148,996.810828
R50_fines,492.0,330.556280,235.993585,64.210862,142.579155,213.098175,580.026235,873.066616
R50_oversize,492.0,146.295647,108.429543,28.072745,58.843228,91.697836,263.430771,414.330461


Saved v2 submission: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_forward_v2_advanced_features.csv
Saved v2 model: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/models/final_forward_v2_advanced_features.joblib


## 17. Save Experiment Reports

This saves the comparison tables so we can include them in the final report.

In [17]:
if RUN_FULL_EVALUATION:
    results_path = REPORTS_DIR / "forward_v2_advanced_features_results.csv"
    constraints_path = REPORTS_DIR / "forward_v2_advanced_features_constraint_metrics.csv"
    decisions_path = REPORTS_DIR / "forward_v2_advanced_features_decision_table.csv"

    results_df.to_csv(results_path, index=False)
    constraint_results_df.to_csv(constraints_path, index=False)
    decision_df.to_csv(decisions_path, index=False)

    print("Saved:", results_path)
    print("Saved:", constraints_path)
    print("Saved:", decisions_path)

Saved: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports/forward_v2_advanced_features_results.csv
Saved: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports/forward_v2_advanced_features_constraint_metrics.csv
Saved: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports/forward_v2_advanced_features_decision_table.csv


## 18. Final Interpretation Guide

After running the notebook, use this interpretation:

- If `v2_direct` improves MAE/RMSE/P95 on distance targets, consider upgrading those targets.
- If `v2_log` improves only MAE but worsens RMSE or P95, keep v1 for competition safety.
- If the decision table suggests no upgrades, keep the current final forward v1 model.
- If only one or two targets improve, create a mixed v2 model only for those targets.

The goal is not to force a v2. The goal is to prove whether v2 is better.